# 🚗 EV Driver Behavior Scoring System
## Notebook 2 — Exploratory Data Analysis & Visualization

We explore the dataset to find:
- How features relate to energy consumption
- Which behaviors waste the most EV range
- Distribution of driver efficiency scores
- Correlation between driving style and kWh/100km

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('ev_driver_dataset.csv')
print(f'✅ Dataset loaded: {df.shape[0]} sessions, {df.shape[1]} features')
df.head()

In [ ]:
# ── Cell 2: Dataset Overview ──────────────────────────────────────────────────
print('=== Shape:', df.shape)
print('\n=== Missing Values:')
print(df.isnull().sum())
print('\n=== Descriptive Statistics:')
df.describe().round(2)

In [ ]:
# ── Cell 3: Driver Grade Distribution ────────────────────────────────────────
grade_order  = ['A - Eco Master','B - Smooth Commuter','C - Average Driver',
                'D - Aggressive Urban','F - Energy Waster']
grade_colors = ['#2196F3','#4CAF50','#FF9800','#FF5722','#F44336']

counts = df['driver_grade'].value_counts().reindex(grade_order, fill_value=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Driver Grade Distribution', fontsize=14, fontweight='bold')

bars = ax1.bar(counts.index, counts.values, color=grade_colors, edgecolor='white', width=0.6)
ax1.set_xlabel('Driver Grade')
ax1.set_ylabel('Number of Sessions')
ax1.set_xticklabels(counts.index, rotation=20, ha='right')
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(val), ha='center', fontsize=10, fontweight='bold')

ax2.pie(counts.values, labels=counts.index, colors=grade_colors,
        autopct='%1.1f%%', startangle=140, pctdistance=0.82)
ax2.set_title('Grade Share (%)')

plt.tight_layout()
plt.savefig('grade_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 4: Correlation Heatmap ───────────────────────────────────────────────
num_cols = ['avg_speed_kmh','std_speed_kmh','regen_braking_ratio',
            'aggression_index','smoothness_score','high_speed_ratio',
            'soc_swing_pct','kwh_per_100km','ev_efficiency_score']

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.4, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nTop correlations with kWh/100km:')
print(corr['kwh_per_100km'].sort_values(ascending=False).drop('kwh_per_100km'))

In [ ]:
# ── Cell 5: kWh/100km by Driver Grade (Box Plot) ─────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
grade_order2 = ['A - Eco Master','B - Smooth Commuter','C - Average Driver',
                'D - Aggressive Urban','F - Energy Waster']
palette = {'A - Eco Master':'#2196F3','B - Smooth Commuter':'#4CAF50',
           'C - Average Driver':'#FF9800','D - Aggressive Urban':'#FF5722',
           'F - Energy Waster':'#F44336'}

sns.boxplot(data=df, x='driver_grade', y='kwh_per_100km',
            order=grade_order2, palette=palette, ax=ax, width=0.5)
ax.set_title('Energy Consumption (kWh/100km) by Driver Grade', fontsize=13, fontweight='bold')
ax.set_xlabel('Driver Grade')
ax.set_ylabel('kWh / 100 km')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.axhline(14, color='gray', linestyle='--', linewidth=1, label='Tata Nexon baseline (14 kWh/100km)')
ax.legend()
plt.tight_layout()
plt.savefig('kwh_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 6: Regen Braking vs Energy Consumption ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Key EV Behavioral Relationships', fontsize=13, fontweight='bold')

# Plot 1: Regen ratio vs kWh
scatter1 = axes[0].scatter(
    df['regen_braking_ratio'], df['kwh_per_100km'],
    c=df['ev_efficiency_score'], cmap='RdYlGn', alpha=0.5, s=18
)
axes[0].set_xlabel('Regenerative Braking Ratio')
axes[0].set_ylabel('Energy Consumption (kWh/100km)')
axes[0].set_title('More Regen → Less Energy Used')
plt.colorbar(scatter1, ax=axes[0], label='EV Score')

# Plot 2: Aggression index vs kWh
scatter2 = axes[1].scatter(
    df['aggression_index'], df['kwh_per_100km'],
    c=df['ev_efficiency_score'], cmap='RdYlGn', alpha=0.5, s=18
)
axes[1].set_xlabel('Driving Aggression Index')
axes[1].set_ylabel('Energy Consumption (kWh/100km)')
axes[1].set_title('More Aggression → More Energy Wasted')
plt.colorbar(scatter2, ax=axes[1], label='EV Score')

plt.tight_layout()
plt.savefig('regen_aggression_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 7: EV Score vs Estimated Range ───────────────────────────────────────
# Tata Nexon EV has 40.5 kWh usable battery → range = 40500 / kwh_per_100km * 100
BATTERY_KWH = 40.5
df['estimated_range_km'] = (BATTERY_KWH / df['kwh_per_100km']) * 100

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(df['ev_efficiency_score'], df['estimated_range_km'],
                c=df['aggression_index'], cmap='RdYlGn_r', alpha=0.5, s=20)
ax.set_xlabel('EV Efficiency Score (0–100)')
ax.set_ylabel('Estimated Range (km) — Tata Nexon EV 40.5 kWh')
ax.set_title('Higher Score → More Range per Charge', fontsize=12, fontweight='bold')
plt.colorbar(sc, ax=ax, label='Aggression Index (higher = worse)')
plt.tight_layout()
plt.savefig('score_vs_range.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best driver range:  {df["estimated_range_km"].max():.0f} km')
print(f'Worst driver range: {df["estimated_range_km"].min():.0f} km')
print(f'Difference:         {df["estimated_range_km"].max() - df["estimated_range_km"].min():.0f} km — just from driving behavior!')

## ✅ Notebook 2 Complete!

### Key Insights Found:
- Higher regen braking ratio strongly reduces kWh/100km
- Aggressive drivers consume significantly more energy per 100 km
- Driving behavior alone can cause 50+ km difference in real-world range
- Smoothness score and regen ratio are the most impactful features

### Next → Notebook 3: ML Model (KMeans + XGBoost + SHAP)